In [2]:
#Imports
from langchain_openai import OpenAIEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [3]:
#Initialization of varibales

topic = {
    "array": 0,
    "dictionary" : 0,
    "linked list": 0,
    "tree": 0,
    "graph": 0
}


user_profile = {
    "topic_strength": topic,
    "learning_style": None
}



In [4]:
#main - chatbased CLI
print("Welcome! Let's personalize your CS study notes!Please answer the following questions.")
inintial_topic_questions =["How comfortable are you with Arrays?",
                     "How comfortable are you with Dictionary?",
                     "How comfortable are you with Linked List?",
                     "How comfortable are you with Tree?",
                     "How comfortable are you with Graph?"]
sample_topic_answers = []

initial_personalization_questions = [
    "Do you prefer direct steps or detailed discussion? Press 1 for direct steps, 2 for detailed discussion",
    "Would you prefer a checklist or conversation? Press 1 for checklist, 2 for conversation"
]

initial_personalization_answers = []
#getting user's input for the initial topic based questions

for i in range(len(inintial_topic_questions)):
    print(inintial_topic_questions[i])
    sample_topic_answers.append(input("Give answer between 1-5"))

for j in range(len(initial_personalization_questions)):
    print(initial_personalization_questions[j])
    initial_personalization_answers.append(input())
    

Welcome! Let's personalize your CS study notes!Please answer the following questions.
How comfortable are you with Arrays?
How comfortable are you with Dictionary?
How comfortable are you with Linked List?
How comfortable are you with Tree?
How comfortable are you with Graph?
Do you prefer direct steps or detailed discussion? Press 1 for direct steps, 2 for detailed discussion
Would you prefer a checklist or conversation? Press 1 for checklist, 2 for conversation


In [5]:
print(initial_personalization_answers)

['2', '2']


In [6]:
#assign initial topic answers to topic dictionary

for key, value in zip(topic, sample_topic_answers):
    topic[key] = int(value)

print(topic)

for k in range(len(initial_personalization_answers)-1):
    if initial_personalization_answers[k] == "1" and initial_personalization_answers[k+1] == "1":
        user_profile["learning_style"] = "action-based"
    elif initial_personalization_answers[k] == "2" and initial_personalization_answers[k+1] == "2":
        user_profile["learning_style"] ="relationship-based"
    else:
        user_profile["learning_style"] = "mixed"

print(user_profile)

{'array': 1, 'dictionary': 2, 'linked list': 3, 'tree': 4, 'graph': 5}
{'topic_strength': {'array': 1, 'dictionary': 2, 'linked list': 3, 'tree': 4, 'graph': 5}, 'learning_style': 'relationship-based'}


In [7]:
#Generate notes based on weak topics

def find_weakest_topic(user_profile):
    min_key = min(user_profile["topic_strength"], key =user_profile["topic_strength"].get)

    return min_key
print(find_weakest_topic(user_profile))
    

array


In [8]:
from dotenv import load_dotenv
import os
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Access the API key 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

In [20]:
#Generates notes based on user's learning style and weak topic
full_prompt= f""""
You are an expert Python-based study note generator for computer science students.
Generate concise and helpful study notes on the user's weakest topic: {find_weakest_topic(user_profile)}.
Adapt the tone and structure based on the user's learning style: {user_profile["learning_style"]}.

Use the following formatting rules based on the learning style:

If learning_style is "action-based", structure the output as:

- Simple, direct sentences
- Concise bullet points
- Task- and outcome-focused content
- Give structure coding example

If learning_style is "relationship-based", structure the output as:

- Simple, direct sentences
- A narrative or dialogue-style explanation
- Paragraph format with emotional/contextual cues to build understanding
- Give structure coding example

Ensure the content remains clear, engaging, and easy to follow regardless of the style.
"""


response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo", or "gpt-4o-mini"
        messages=[
            {"role": "user",
            "content": full_prompt},
        ]
    )

    # Print the response text
response_text = response.choices[0].message.content
print(response_text)

**Study Notes on Arrays for Relationship-Based Learners**

Imagine you’re at a party, and each guest represents an element in an array. Just like how you might arrange guests based on certain criteria—maybe by their height or their age—arrays allow us to organize data in a specific sequence. This organization helps us keep track of information and access it easily.

An array is simply a collection of items stored at contiguous memory locations. Essentially, it allows you to group related data together. For instance, think of your favorite playlist where each song is stored in a certain order. If you want to access a specific song, you just need to know what position it's in. In programming, we do this using indices, starting from zero for the first item.

Let's explore arrays through a practical lens. Imagine you're creating a simple program to store your favorite fruits. You can create an array that holds the names of these fruits:

```python
# Creating an array of fruits
fruits = ["a

In [8]:
#Note feedback loop

survey =[
    {
        "id": 1,
        "question": "How helpful was this note?",
        "options": ["Very helpful and clear", "Somewhat helpful and could be clearer", "Not helpful"]
    },
    {
        "id": 2,
        "question": "How would you describe this note?",
        "options": ["Straight forward and to the point", "Detailed and storylike"]
    },
    {
        "id": 3,
        "question": "What would you prefer more in this note?",
        "options": ["More step by step instructions", "More background, context, stories"]
    },
    {
        "id": 4,
        "question": "Overall does this note match your learning style?",
        "options": ["Perfect match", "Needs more details, stories, example", "Needs more concise example"]
    }
]

In [9]:
#Note survey function
response = []
for i in range(len(survey)):
    print(survey[i]["question"])
    for j in range (len(survey[i]["options"])):
        print(f"{j+1}. {survey[i]["options"][j]}")
    response.append(input("Press only 1 desired number"))


How helpful was this note?
1. Very helpful and clear
2. Somewhat helpful and could be clearer
3. Not helpful
How would you describe this note?
1. Straight forward and to the point
2. Detailed and storylike
What would you prefer more in this note?
1. More step by step instructions
2. More background, context, stories
Overall does this note match your learning style?
1. Perfect match
2. Needs more details, stories, example
3. Needs more concise example


In [21]:
from llama_index.core.node_parser import (
    SentenceSplitter,
    SemanticSplitterNodeParser,
)
from llama_index.embeddings.openai import OpenAIEmbedding

In [22]:
embed_model = OpenAIEmbedding()
splitter = SemanticSplitterNodeParser(
    buffer_size=5, breakpoint_percentile_threshold=30, embed_model=embed_model
)


In [23]:
from llama_index.core import Document
doc = Document(text=response_text)

In [24]:
nodes = splitter.get_nodes_from_documents([doc]) 

In [40]:
print(nodes[6].get_content())

You can add new guests, just like adding elements to your array using the `append` method. 


In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=500,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

In [46]:
texts = text_splitter.create_documents([response_text])

In [53]:
print(texts[3])
len(texts)

page_content='# Adding a fruit to the list
fruits.append("elderberry")
print(fruits)  # Output: ['apple', 'banana', 'cherry', 'date', 'elderberry']
```

In the code above, each fruit is like a guest at the party, and you can find them by knowing their order. You can add new guests, just like adding elements to your array using the `append` method. This method is a wonderful way to grow your array, just as you might invite new friends to your gathering.'


5

In [10]:
#add faiss to the chunks

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [56]:
#get whats the format for texts

library = FAISS.from_documents(texts, embeddings)

In [57]:
#save faiss index

library.save_local("faiss_index_relationship_chunks")

In [11]:
r_chunk_saved = FAISS.load_local("faiss_index_relationship_chunks", embeddings, allow_dangerous_deserialization=True)

In [ ]:
response_text = ""

for faiss_id in range(r_chunk_saved.index.ntotal):
    # Get the docstore ID
    docstore_id = r_chunk_saved.index_to_docstore_id[faiss_id]
    doc = r_chunk_saved.docstore._dict[docstore_id]
    chunk_text = doc.page_content

    # Prompt for quiz generation
    prompt = f"""
    Based on the following text chunk, generate 2-5 multiple choice questions
    with answers and explanations. Minimum 2 questions, maximum 5 based on relavance. 
    
    - Do not focus on the analogy. Make questions based on programming concept present in the chunk.
    - If you're asking coding based question, make sure to give the relavant code before asking. For example, if you ask what is the output of fruits[0] then surely give fruits array. 
    - Don't make the answer choices obvious. Focus on programming concepts.  
    
    For each question, provide the correct answer immediately after the question, labeled with "ANSWER:", and then provide the explanation labeled with "EXPLANATION:".

    Example format:
    1. What is the output of the following code?
    A) Option A
    B) Option B
    C) Option C
    D) Option D

    ANSWER: B

    EXPLANATION: This is why B is the correct answer.

    Text chunk:
    {chunk_text}
    """

    # Call OpenAI
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a Python language based Computer Science quiz generator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )

    # Print the result
    response_text += f"\n\nChunk {faiss_id}\n\n"
    response_text += response.choices[0].message.content
print(response_text)



=== Quiz for Chunk 0 ===

1. What is the primary purpose of using arrays in programming?

A) To store data in a random order  
B) To organize data in a specific sequence  
C) To perform mathematical operations  
D) To create complex data types  

ANSWER: B

EXPLANATION: Arrays are used to organize data in a specific sequence, allowing for easy access and management of information. This is analogous to arranging guests at a party based on certain criteria, where the order matters.

2. Given the following code snippet, what will be the output of `guests[2]`?

```python
guests = ["Alice", "Bob", "Charlie", "Diana"]
```

A) Alice  
B) Bob  
C) Charlie  
D) Diana  

ANSWER: C

EXPLANATION: In Python, list indexing starts at 0. Therefore, `guests[2]` refers to the third element in the list, which is "Charlie". 

3. If you want to add a new guest "Eve" to the end of the list of guests, which of the following code snippets will achieve that?

A) `guests.append("Eve")`  
B) `guests.add("Eve")

In [10]:
response_text = f"""
Chunk 0

1. What is the primary purpose of using arrays in programming?

A) To store data in a random order  
B) To organize data in a specific sequence  
C) To perform mathematical operations  
D) To create complex data types  

ANSWER: B

EXPLANATION: Arrays are used to organize data in a specific sequence, allowing for easy access and management of information. This is analogous to arranging guests at a party based on certain criteria, where the order matters.

2. Given the following code snippet, what will be the output of `guests[2]`?

```python
guests = ["Alice", "Bob", "Charlie", "Diana"]
```

A) Alice  
B) Bob  
C) Charlie  
D) Diana  

ANSWER: C

EXPLANATION: In Python, list indexing starts at 0. Therefore, `guests[2]` refers to the third element in the list, which is "Charlie". 

3. If you want to add a new guest "Eve" to the end of the list of guests, which of the following code snippets will achieve that?

A) `guests.append("Eve")`  
B) `guests.add("Eve")`  
C) `guests.insert("Eve", 4)`  
D) `guests[4] = "Eve"`  

ANSWER: A

EXPLANATION: The `append()` method is used to add an element to the end of a list in Python. Options B and C do not exist for lists, and D would result in an IndexError since the index 4 does not currently exist.

4. What will happen if you try to access `guests[4]` in the following code?

```python
guests = ["Alice", "Bob", "Charlie", "Diana"]
```

A) It will return "None"  
B) It will return an error  
C) It will return an empty string  
D) It will return "Eve"  

ANSWER: B

EXPLANATION: Attempting to access `guests[4]` will result in an IndexError because the list only contains four elements (indices 0 to 3). There is no fifth element (index 4) in the list.

Chunk 1

1. What will be the output of the following code?
```python
playlist = ["Song A", "Song B", "Song C", "Song D"]
print(playlist[2])
```
A) Song A  
B) Song B  
C) Song C  
D) Song D  

ANSWER: C

EXPLANATION: In Python, array (or list) indexing starts at zero. Thus, `playlist[2]` accesses the third item in the list, which is "Song C".

2. In the context of arrays, which statement is true regarding memory allocation?
A) Arrays can store any type of data without restrictions.  
B) Arrays must be of fixed size and allocate memory for all elements at once.  
C) Arrays dynamically resize themselves based on the number of elements stored.  
D) Arrays store data in non-contiguous memory locations.  

ANSWER: B

EXPLANATION: Arrays, by definition, allocate a contiguous block of memory for a fixed number of elements. This means that the size of the array must be defined at the time of creation, and all memory for the elements is allocated at once.

3. Given the following code snippet, what will be the result of the expression `len(playlist)`?
```python
playlist = ["Song A", "Song B", "Song C", "Song D"]
```
A) 3  
B) 4  
C) 5  
D) None  

ANSWER: B

EXPLANATION: The `len()` function in Python returns the number of items in a list. Since there are four songs in the `playlist` list, `len(playlist)` will return 4.

4. If you wanted to add a new song "Song E" to the end of the `playlist`, which code would accomplish that?
A) `playlist.add("Song E")`  
B) `playlist.append("Song E")`  
C) `playlist.insert("Song E", 4)`  
D) `playlist.extend("Song E")`  

ANSWER: B

EXPLANATION: The `append()` method is used to add an item to the end of a list in Python. Therefore, `playlist.append("Song E")` correctly adds "Song E" to the end of the `playlist`.

Chunk 2

1. What is the output of the following code?
```python
fruits = ["apple", "banana", "cherry", "date"]
print(fruits[1])
```
A) apple  
B) banana  
C) cherry  
D) date  

ANSWER: B  

EXPLANATION: In Python, arrays (or lists) are zero-indexed, meaning the first element is accessed with index 0. Therefore, `fruits[1]` refers to the second element in the list, which is "banana".

2. What will be the result of the following code?
```python
fruits = ["apple", "banana", "cherry", "date"]
fruits[2] = "blueberry"
print(fruits)
```
A) ['apple', 'banana', 'blueberry', 'date']  
B) ['apple', 'banana', 'cherry', 'blueberry']  
C) ['apple', 'blueberry', 'cherry', 'date']  
D) ['blueberry', 'banana', 'cherry', 'date']  

ANSWER: A  

EXPLANATION: The code updates the element at index 2 of the `fruits` list from "cherry" to "blueberry". Therefore, when printed, the list reflects this change: ['apple', 'banana', 'blueberry', 'date'].

3. If you wanted to retrieve the last fruit in the array, which of the following code snippets would you use?
A) `fruits[-1]`  
B) `fruits[len(fruits)]`  
C) `fruits[len(fruits)-1]`  
D) Both A and C  

ANSWER: D  

EXPLANATION: In Python, negative indexing allows you to access elements from the end of the list. `fruits[-1]` retrieves the last item, and `fruits[len(fruits)-1]` also correctly accesses the last item by calculating the index. Thus, both options A and C are valid ways to access the last fruit in the array.

Chunk 3

1. What will be the output of the following code?
```python
fruits = ['apple', 'banana', 'cherry', 'date']
fruits.append("elderberry")
print(fruits)
```
A) ['apple', 'banana', 'cherry', 'date']  
B) ['apple', 'banana', 'cherry', 'date', 'elderberry']  
C) ['elderberry', 'apple', 'banana', 'cherry', 'date']  
D) ['apple', 'banana', 'elderberry', 'cherry', 'date']  

ANSWER: B

EXPLANATION: The `append` method adds "elderberry" to the end of the `fruits` list. As a result, when printed, the list shows all previous fruits plus the new addition: ['apple', 'banana', 'cherry', 'date', 'elderberry'].

2. What is the time complexity of the `append` method in Python lists?
A) O(1)  
B) O(n)  
C) O(log n)  
D) O(n^2)  

ANSWER: A

EXPLANATION: The `append` method has an average time complexity of O(1), meaning it takes constant time to add an element to the end of the list, regardless of the size of the list.

3. If the following code is executed, what would be the value of `fruits[0]`?
```python
fruits = ['apple', 'banana', 'cherry', 'date']
fruits.append("elderberry")
```
A) 'banana'  
B) 'cherry'  
C) 'apple'  
D) 'date'  

ANSWER: C

EXPLANATION: The first element of the list `fruits` is 'apple', which is at index 0. The `append` method adds "elderberry" to the end of the list but does not change the existing order or values of the elements already in the list.

4. Which method can be used to remove the last item from the `fruits` list?
A) fruits.remove()  
B) fruits.pop()  
C) fruits.delete()  
D) fruits.clear()  

ANSWER: B

EXPLANATION: The `pop()` method removes the last element from a list and returns it. In contrast, `remove()` is used to remove a specific item by its value, `delete()` is not a valid list method in Python, and `clear()` removes all items from the list.

Chunk 4

1. Which of the following best describes an array in programming?
A) A single data type that cannot store multiple values
B) A collection of elements that can be of different data types
C) A structured collection of elements that are indexed and usually of the same data type
D) A method to store only string values

ANSWER: C

EXPLANATION: An array is a structured collection of elements, typically of the same data type, that are indexed. This allows for organized storage and easy access to the data elements based on their indices.

2. What is a common use case for arrays in programming?
A) To create complex algorithms only
B) To store a single value at a time
C) To manage collections of related data efficiently
D) To optimize the speed of program execution significantly

ANSWER: C

EXPLANATION: Arrays are commonly used to manage collections of related data efficiently. They allow programmers to store multiple values in a single variable, making it easier to organize and manipulate data.
"""

In [13]:
import re
inner_dictionary = {
    "question": None,
    "options": [],
    "answer": None,
    "explanation": None 
}

outer_dictionary = {} #add chunk numebr here 

for lines in response_text.splitlines():
    line = lines.strip()
    
    if line.startswith("Chunk"):
        current_chunk_id = line
        outer_dictionary[current_chunk_id] = {}
        continue
        
    match = re.match(r'^\d+\.', line)
    
    if match:
        question_number = match.group(0)[:-1]
        
        inner_dictionary= {
            "question": line.replace(match.group(0), "").strip(),
            "options": [],
            "answer": None,
            "explanation": None 
        }
        
        if current_chunk_id:
            outer_dictionary[current_chunk_id][question_number] = inner_dictionary
            current_question_id = question_number
        continue
        
    if line.startswith(("A)", "B)", "C)", "D)", "E)")):
        if current_chunk_id and current_question_id:
            outer_dictionary[current_chunk_id][current_question_id]["options"].append(line)
        continue
        
    if line.startswith("ANSWER:"):
        if current_chunk_id and current_question_id:
            answer_text = line.split(":", 1)[1].strip()
            outer_dictionary[current_chunk_id][current_question_id]["answer"] = answer_text
        continue
        
    if line.startswith("EXPLANATION:"):
        if current_chunk_id and current_question_id:
            explanation_text = line.split(":", 1)[1].strip()
            outer_dictionary[current_chunk_id][current_question_id]["explanation"] = explanation_text
        continue
        
        
print(outer_dictionary)




{'Chunk 0': {'1': {'question': 'What is the primary purpose of using arrays in programming?', 'options': ['A) To store data in a random order', 'B) To organize data in a specific sequence', 'C) To perform mathematical operations', 'D) To create complex data types'], 'answer': 'B', 'explanation': 'Arrays are used to organize data in a specific sequence, allowing for easy access and management of information. This is analogous to arranging guests at a party based on certain criteria, where the order matters.'}, '2': {'question': 'Given the following code snippet, what will be the output of `guests[2]`?', 'options': ['A) Alice', 'B) Bob', 'C) Charlie', 'D) Diana'], 'answer': 'C', 'explanation': 'In Python, list indexing starts at 0. Therefore, `guests[2]` refers to the third element in the list, which is "Charlie".'}, '3': {'question': 'If you want to add a new guest "Eve" to the end of the list of guests, which of the following code snippets will achieve that?', 'options': ['A) `guests.a

['B', 'B', 'C', 'B', 'A', 'A', 'B', 'C', 'B', 'C', 'C', 'B', 'A', 'B', 'C', 'B', 'B', 'B', 'B', 'C', 'B', 'B', 'B']
['Arrays can indeed be used to build complex data structures as they can hold multiple elements that can represent a variety of data types and relationships. They are foundational in programming, allowing for the organization and management of data effectively.', 'In Python, array (or list) indexing starts at 0. Therefore, `fruits[1]` refers to the second element in the `fruits` list, which is "banana".', 'In Python, attempting to access an index that is outside the bounds of an array (or list) results in an IndexError. In this case, `numbers[4]` is out of range since the valid indices for `numbers` are 0 through 3.1. Given the following code snippet, what will be the output of `guests[1]`?', 'In Python, arrays (or lists) are zero-indexed, meaning the first element is at index 0. Therefore, `guests[1]` refers to the second element in the list, which is "Bob".', "In Python